# OpenFold Single Complex

This notebook is for one prediction run: one protein or one complex.

Use it when you want to answer: `what does this one system look like?`

Limits:
- confidence metrics are not direct ddG estimates
- ipTM is useful for ranking interface candidates, not for binding affinity claims


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
HELPERS_DIR = PROJECT_ROOT / 'helpers'
if str(HELPERS_DIR) not in sys.path:
    sys.path.insert(0, str(HELPERS_DIR))

import pandas as pd
from of_notebook_lib import (
    RuntimeConfig,
    format_sample_table,
    preview_molecules,
    run_single_case,
    summarize_best_result,
    validate_molecules,
)

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)


## Runtime Setup

Change these paths only if your OpenFold environment lives somewhere else.

In [ ]:
runtime = RuntimeConfig(
    project_dir=Path('/home/jovyan/OpenFold'),
    openfold_prefix=Path('/home/jovyan/.mlspace/envs/openfold310'),
    results_dir=Path('/home/jovyan/OpenFold/results_refactored'),
    msa_cache_dir=Path('/home/jovyan/OpenFold/msa_cache/colabfold_msas'),
    triton_cache_dir=Path('/tmp/triton_cache'),
    fixed_msa_tmp_dir=Path('/tmp/of3_colabfold_msas'),
    use_fused_attention=False,
    use_deepspeed=False,
)

runtime


## User Input

Edit only this cell for normal usage.

In [ ]:
experiment_name = 'single_pair_test'

molecules = [
    {
        'molecule_type': 'protein',
        'chain_ids': ['A'],
        'sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG',
    },
    {
        'molecule_type': 'protein',
        'chain_ids': ['B'],
        'sequence': 'MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG',
    },
]

mutation = {
    'enabled': False,
    'chain_id': 'B',
    'position_1based': 10,
    'new_residue': 'A',
}

use_templates = True
use_msa_server = True
num_diffusion_samples = 1
num_model_seeds = 1
runner_yaml = None
inference_ckpt_path = None
inference_ckpt_name = None


## Input Preview

In [ ]:
issues = validate_molecules(molecules)
if issues:
    for issue in issues:
        print('INPUT ISSUE:', issue)
else:
    print('Input looks valid.')

display(preview_molecules(molecules))
print('Mutation config:', mutation)


## Run Prediction

In [ ]:
result = run_single_case(
    runtime=runtime,
    experiment_name=experiment_name,
    molecules=molecules,
    mutation=mutation,
    use_templates=use_templates,
    use_msa_server=use_msa_server,
    num_diffusion_samples=num_diffusion_samples,
    num_model_seeds=num_model_seeds,
    runner_yaml=runner_yaml,
    inference_ckpt_path=inference_ckpt_path,
    inference_ckpt_name=inference_ckpt_name,
)

print('Return code:', result.return_code)
print('Run dir:', result.run_dir)
print('Summary dir:', result.summary_dir)


## Best Samples Table

In [ ]:
display(format_sample_table(result.samples_df))


## Quick Interpretation

In [ ]:
summary = summarize_best_result(result.samples_df)
for key, value in summary.items():
    print(f'{key}: {value}')


## Key Files

In [ ]:
print('query.json        ->', result.query_path)
print('openfold log      ->', result.log_path)
print('output directory  ->', result.output_dir)
print('summary directory ->', result.summary_dir)
print('best report       ->', result.summary_dir / 'best_samples_report.txt')
